In [1]:
%load_ext autoreload
%autoreload 2

import image_preparation as prep

## Image preparation (web and low res, without watermark)

In [7]:
source_dir = "./../images_to_convert"
low_res_dir = './../images_converted/images_low_res'
result_dir="./../images_converted/images_web"

In [8]:
prep.prepare_images_in_dir(
    source_dir=source_dir,
    result_dir=result_dir,
    max_width=1700,
    quality=90,
)

Processing: ./../images_to_convert\concrete_commute.jpg
- Removing alpha channel
- Scaling image with size: (2914, 3711)
Web export saved to './../images_converted/images_web\concrete_commute.jpg' (1334x1700, q90)


In [9]:
# low res

result_img_max_width=800
ext = ['png', 'jpg', "jpeg"]
low_res_max_width = 400
suffix = None
overwrite=False

# scale image
prep.scale_images_in_dir(source_dir, low_res_dir, low_res_max_width, suffix, overwrite)

Scaling image ./../images_converted/images_low_res\concrete_commute.jpg
Processing: ./../images_to_convert\concrete_commute.jpg
- Scaling image with size: (2914, 3711)
- Removing alpha channel
Processed image saved to './../images_converted/images_low_res\concrete_commute.jpg'
--------------------------------------------------


## Image preparation watermark

In [ ]:
watermark_path = './../src/images/logo/watermark.png'
black_watermark_path = './../src/images/logo/watermark_black.png'

# all images
# source_dir = './../src/assets/artwork/images'
source_dir = "./../to_convert"
# watermark target dir
watermark_dir = "./../converted" # './../src/assets/artwork/images_watermark'

low_res_dir = './../converted/images_low_res'

In [9]:
watermark_width=150
position = 'bottom_right'  # Available positions: 'top_left', 'top_right', 'bottom_left', 'bottom_right', 'center'
opacity = 0.4
padding=25
result_img_max_width=800
ext = ['png', 'jpg', "jpeg"]
low_res_max_width = 400
suffix = None
overwrite=False

position_black = 'center'  # Available positions: 'top_left', 'top_right', 'bottom_left', 'bottom_right', 'center'
opacity_black = 0.25
watermark_width_black = 300

In [10]:
# scale image
prep.scale_images_in_dir(source_dir, low_res_dir, low_res_max_width, suffix, True)

Scaling image ./../converted/images_low_res\bushwick_beauty.jpg
Processing: ./../to_convert\bushwick_beauty.jpg
- Scaling image with size: (2886, 3674)
- Removing alpha channel
Processed image saved to './../converted/images_low_res\bushwick_beauty.jpg'
--------------------------------------------------


In [11]:
prep.watermark_images(source_dir, 
        watermark_dir, 
        watermark_path, 
        watermark_width=watermark_width,
        padding=padding,
        position = position,  # Available positions: 'top_left', 'top_right', 'bottom_left', 'bottom_right', 'center'
        opacity = opacity,
        result_img_max_width=result_img_max_width,
        ext = ext,
        suffix=suffix,
        overwrite=overwrite)


Loading Watermark from path './../src/images/logo/watermark.png'
Rescaling watermark to width: 150
Loading images from './../to_convert' with extensions: ['png', 'jpg', 'jpeg']
Saving them to './../converted'
Processing: ./../to_convert\bushwick_beauty.jpg
- Changing watermark opacity to 0.4
- Scaling image with size: (2886, 3674)
- Removing alpha channel
Processed image saved to './../converted\bushwick_beauty.jpg'
--------------------------------------------------


In [ ]:
## for sketches, use black watermark
# all images
sketches_source_dir = './../src/assets/artwork/images/sketches'
# watermark target dir
sketches_watermark_dir = './../src/assets/artwork/images_watermark/sketches'

prep.watermark_images(sketches_source_dir, 
        sketches_watermark_dir, 
        black_watermark_path, 
        watermark_width=watermark_width_black,
        padding=padding,
        position = position,  # Available positions: 'top_left', 'top_right', 'bottom_left', 'bottom_right', 'center'
        opacity = opacity,
        result_img_max_width=result_img_max_width,
        ext = ext,
        suffix=suffix,
        overwrite=overwrite)


Loading Watermark from path './../src/images/logo/watermark_black.png'
Rescaling watermark to width: 300
Loading images from './../src/assets/artwork/images/sketches' with extensions: ['png', 'jpg', 'jpeg']
Saving them to './../src/assets/artwork/images_watermarked/sketches'


In [13]:
# verify all markdown files have image

markdown_folder = './../src/assets/artwork/mdfiles'  # Replace this with the actual folder path
prep.check_images_exist(markdown_folder)

File not found: ..\src\assets\artwork\images_watermarked\black-and-white\bushwick_beauty.jpg
File not found: ..\src\assets\artwork\images_low_res\black-and-white\bushwick_beauty.jpg


# Upscale Image

In [ ]:
filename = "dom2_high_contrast.png"
img = Image.open(filename)

# 1. Define the physical target size you need at 300 DPI
# Example: You want a 4x6 inch print
target_width_inches = 297 / 25.4
target_height_inches = 420 / 25.4
target_dpi = 320

# 2. Calculate the required pixel grid size
new_width = round(target_width_inches * target_dpi)
new_height = round(target_height_inches * target_dpi)


# 3. Resize using a high-quality resampling filter (LANCZOS)
resized_img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)

# 4. JPEG has no alpha channel, so flatten RGBA/P images onto a white background
if resized_img.mode in ("RGBA", "P"):
    resized_img = resized_img.convert("RGBA")
    background = Image.new("RGB", resized_img.size, (255, 255, 255))
    background.paste(resized_img, mask=resized_img.split()[3])
    resized_img = background

# 5. Save the file with the new pixels AND the matching metadata
resized_img.save(f"{filename.split('.')[0]}_{target_dpi}dpi.jpg", dpi=(target_dpi, target_dpi))


# Other stuff

In [66]:
from PIL import Image
# change watermark color

def replace_white_with_gray(input_path, output_path):
    # Load the input PNG image
    try:
        img = Image.open(input_path)
    except IOError:
        print(f"Error: Could not open the image at {input_path}")
        return

    # Convert the image to RGBA mode to handle transparency (if any)
    img = img.convert("RGBA")

    # Get the image dimensions
    width, height = img.size

    # Define the color for the gray pixels (e.g., RGB value of (128, 128, 128))
    gray_color = (0, 0, 0, 255)  # R, G, B, A

    # Process each pixel and replace white ones with gray
    for x in range(width):
        for y in range(height):
            pixel_color = img.getpixel((x, y))
            if pixel_color[:3] == (255, 255, 255):  # Check if it's a white pixel (R, G, B values)
                img.putpixel((x, y), gray_color)

    # Save the modified image
    try:
        img.save(output_path)
        print(f"Image saved to {output_path}")
    except IOError:
        print(f"Error: Could not save the image to {output_path}")

# Example usage:
output_image_path = "output.png"
replace_white_with_gray(watermark_path, black_watermark_path)


Image saved to ./../src/images/logo/watermark_black.png


In [69]:
import os
overwrite=True
watermark_width=300
opacity=0.3
position="bottom_right"
padding=100

image_file = "./../to_convert/dom1.png"
#"./../src/assets/artwork/images/colorful/hills_of_rio_small.jpg"
result_path = "./../converted/dom1.png"

watermark = Image.open(black_watermark_path).convert("RGBA")

print(f"Rescaling watermark to width: {watermark_width}")
# Calculate the proportional height for the second image based on the fixed width (250px)
width_percent = (watermark_width / float(watermark.size[0]))
new_height = int((float(watermark.size[1]) * float(width_percent)))
# Resize the second image to the calculated proportional size
watermark = watermark.resize(
    (watermark_width, new_height), Image.LANCZOS)

# If the output file already exists, delete it before saving the new image
if not overwrite and os.path.exists(result_path):
    print(f"- Not overwriting existing image '{result_path}'")
else:
    # Process each image file here
    print("Processing:", image_file)
    img = Image.open(image_file).convert("RGBA")
    result_image = prep.merge_images_with_opacity(
        img=img,
        watermark=watermark,
        padding=padding,
        position=position,
        opacity=opacity,
        max_width=result_img_max_width)
    prep.save_image(img=result_image, result_path=result_path,
            suffix=suffix)

Rescaling watermark to width: 300
Processing: ./../to_convert/dom1.png
- Changing watermark opacity to 0.3
- Scaling image with size: (1570, 2200)
- Removing alpha channel
Processed image saved to './../converted/dom1.png'
--------------------------------------------------
